# 4. PD model dataset

| | |
|---|---|
| **Aim** | Build a dataset for the PD model: target variable and explanatory variables |
| **Input data** | `default/` from notebook 2 |
| **Result** | `model_dataset.parquet` |

## Dataset

A snapshot is included in the dataset when:
1. the loan is active and has not yet defaulted;
2. we have a full 12 months of observations after the snapshot.

**Target variable:** `target_12m` = 1 if the first default occurs within the next 12 months, and 0 otherwise.

**No information leakage:** explanatory variables use only data known on the snapshot date. The dataset does not include the date or reason for the default.

**Assumptions to be described in the paper:**
- a loan repaid within the 12-month window has `target_12m` = 0 (competing risk). The `prepaid_12m` flag is used only for analysis and cannot be included in the model;
- the absence of `MI_PCT` is treated as lack of insurance;
- a single loan appears in multiple snapshots, so the observations are not independent

In [1]:
import polars as pl

import config as C
import functions as F

pl.Config.set_tbl_rows(60)
pl.Config.set_tbl_cols(20)
pl.Config.set_fmt_str_lengths(70)
pl.Config.set_tbl_hide_dataframe_shape(True)
pl.Config.set_tbl_hide_column_data_types(True)
C.REPORT_DIR.mkdir(parents=True, exist_ok=True)

## 1. Snapshots for each cohort

In [2]:
C.MODEL_DIR.mkdir(parents=True, exist_ok=True)
paths = sorted(C.DEFAULT_DIR.glob("*.parquet"))
data_end = pl.scan_parquet(paths).select(pl.col("ACT_PERIOD").max()).collect().item()
print(f"Data end: {data_end}")

summary = []
for path in paths:
    model = F.build_model_dataset(pl.read_parquet(path), data_end)
    model.write_parquet(C.MODEL_DIR / path.name)
    summary.append({"cohort": path.stem, "snapshots": model.height, "loans": model["loan_id"].n_unique(),
                    "target_1": int(model["target_12m"].sum()),
                    "target_rate_pct": round(float(model["target_12m"].mean()) * 100, 2)})
pl.DataFrame(summary)

Data end: 2026-06-01


cohort,snapshots,loans,target_1,target_rate_pct
"""2007Q1""",116368,6311,4519,3.88
"""2019Q1""",88256,8687,981,1.11


## 2. One dataset

In [3]:
pl.scan_parquet(C.MODEL_DIR / "*.parquet").sink_parquet(C.MODEL_FILE)
data = pl.scan_parquet(C.MODEL_FILE)
data.select(
    rows=pl.len(), loans=pl.col("loan_id").n_unique(),
    first_snapshot=pl.col("snapshot_date").min(), last_snapshot=pl.col("snapshot_date").max(),
    target_1=pl.col("target_12m").sum(),
    target_rate_pct=(pl.col("target_12m").mean() * 100).round(2),
).collect()

rows,loans,first_snapshot,last_snapshot,target_1,target_rate_pct
204624,14998,2007-03-01,2025-06-01,5500,2.69


## 3. Validation

In [4]:
panel = pl.scan_parquet(C.DEFAULT_DIR / "*.parquet")
defaults = panel.group_by(loan_id=pl.col("LOAN_ID")).agg(default_date=pl.col("first_default_date").first())
checked = data.join(defaults, on="loan_id")
snapshot = pl.col("snapshot_date")
window_end = snapshot.dt.offset_by(f"{C.TARGET_HORIZON_M}mo")
default_date = pl.col("default_date")
in_window = ((default_date > snapshot) & (default_date <= window_end)).fill_null(False)

quality = checked.select(
    target_1_outside_window=((pl.col("target_12m") == 1) & ~in_window).sum(),
    target_0_inside_window=((pl.col("target_12m") == 0) & in_window).sum(),
    default_before_snapshot=(default_date <= snapshot).sum(),
    duplicates=pl.len() - pl.struct("loan_id", "snapshot_date").n_unique(),
    window_after_data_end=(window_end > data_end).sum(),
).collect()
assert all(value == 0 for value in quality.row(0)), quality
quality

target_1_outside_window,target_0_inside_window,default_before_snapshot,duplicates,window_after_data_end
0,0,0,0,0


## 4. Variables

- **Structural:** Known on the date of origination, they do not change over time.
- **Cyclical:** change over time and depend on the borrower's situation or the economy.
- **Mied:** combine loan characteristics with the passage of time.

The dataset does not yet contain macroeconomic data (real estate prices, unemployment, market rate) or current LTV. These will be added in the next step, once they are downloaded.

In [5]:
dictionary = F.variable_dictionary()
dictionary.write_csv(C.REPORT_DIR / "model_variables.csv")
dictionary

variable,group,description
"""fico""","""structural""","""lower of the borrowers' FICO scores at origination"""
"""fico_missing""","""structural""","""1 if FICO is missing"""
"""ltv_orig""","""structural""","""LTV at origination (%)"""
"""cltv_orig""","""structural""","""CLTV at origination (%)"""
"""dti""","""structural""","""DTI at origination (%)"""
"""dti_missing""","""structural""","""1 if DTI is missing"""
"""rate_orig""","""structural""","""interest rate at origination (%)"""
"""loan_amount_k""","""structural""","""loan amount (thousand USD)"""
"""mi_pct""","""structural""","""insurance coverage (%), missing treated as 0"""
"""n_borrowers""","""structural""","""number of borrowers"""
